<a href="https://colab.research.google.com/github/NegasaReta/Elevvo-Internship-Machine-Learning-Track/blob/Task1/Bonus2_student_score_prediction.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

##Important Libraries

In [1]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split
from sklearn.pipeline import Pipeline
from sklearn.compose import ColumnTransformer
from sklearn.preprocessing import OneHotEncoder, StandardScaler
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_absolute_error, mean_squared_error, r2_score


##Load the DataSet

In [3]:
CSV_PATH = "/content/StudentPerformanceFactors.csv"
TARGET_COL = "Exam_Score"

df = pd.read_csv(CSV_PATH)

In [6]:
df.head()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
0,23,84,Low,High,No,7,73,Low,Yes,0,Low,Medium,Public,Positive,3,No,High School,Near,Male,67
1,19,64,Low,Medium,No,8,59,Low,Yes,2,Medium,Medium,Public,Negative,4,No,College,Moderate,Female,61
2,24,98,Medium,Medium,Yes,7,91,Medium,Yes,2,Medium,Medium,Public,Neutral,4,No,Postgraduate,Near,Male,74
3,29,89,Low,Medium,Yes,8,98,Medium,Yes,1,Medium,Medium,Public,Negative,4,No,High School,Moderate,Male,71
4,19,92,Medium,Medium,Yes,6,65,Medium,Yes,3,Medium,High,Public,Neutral,4,No,College,Near,Female,70


In [7]:
df.tail()

,Hours_Studied,Attendance,Parental_Involvement,Access_to_Resources,Extracurricular_Activities,Sleep_Hours,Previous_Scores,Motivation_Level,Internet_Access,Tutoring_Sessions,Family_Income,Teacher_Quality,School_Type,Peer_Influence,Physical_Activity,Learning_Disabilities,Parental_Education_Level,Distance_from_Home,Gender,Exam_Score
6602,25,69,High,Medium,No,7,76,Medium,Yes,1,High,Medium,Public,Positive,2,No,High School,Near,Female,68
6603,23,76,High,Medium,No,8,81,Medium,Yes,3,Low,High,Public,Positive,2,No,High School,Near,Female,69
6604,20,90,Medium,Low,Yes,6,65,Low,Yes,3,Low,Medium,Public,Negative,2,No,Postgraduate,Near,Female,68
6605,10,86,High,High,Yes,6,91,High,Yes,2,Low,Medium,Private,Positive,3,No,High School,Far,Female,68
6606,15,67,Medium,Low,Yes,9,94,Medium,Yes,0,Medium,Medium,Public,Positive,4,No,Postgraduate,Near,Male,64


In [5]:
df.shape

(6607, 20)

In [8]:
##Drop rows where target is missing

In [9]:
df[TARGET_COL] = pd.to_numeric(df[TARGET_COL], errors="coerce")
df = df.dropna(subset=[TARGET_COL])

In [10]:
def evaluate_feature_set(feature_cols, df=df):
    X = df[feature_cols].copy()
    y = df[TARGET_COL].values

    # Identify column types
    numeric_features = X.select_dtypes(include=["number"]).columns.tolist()
    categorical_features = [c for c in X.columns if c not in numeric_features]

    numeric_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler())
    ])

    categorical_transformer = Pipeline(steps=[
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("onehot", OneHotEncoder(handle_unknown="ignore"))
    ])

    preprocessor = ColumnTransformer(
        transformers=[
            ("num", numeric_transformer, numeric_features),
            ("cat", categorical_transformer, categorical_features),
        ],
        remainder="drop"
    )

    model = Pipeline(steps=[
        ("preprocess", preprocessor),
        ("lr", LinearRegression())
    ])

    X_train, X_test, y_train, y_test = train_test_split(
        X, y, test_size=0.2, random_state=42
    )

    model.fit(X_train, y_train)
    y_pred = model.predict(X_test)

    mae = mean_absolute_error(y_test, y_pred)
    mse = mean_squared_error(y_test, y_pred)
    rmse = np.sqrt(mse)
    r2 = r2_score(y_test, y_pred)
    return mae, mse, rmse, r2

##Define feature combinations to test

In [11]:
feature_sets = {
    "Baseline_hours_only": [
        "Hours_Studied"
    ],
    "Hours_plus_previous_scores": [
        "Hours_Studied", "Previous_Scores"
    ],
    "Core_academic": [
        "Hours_Studied", "Previous_Scores", "Attendance", "Tutoring_Sessions"
    ],
    "Lifestyle_added": [
        "Hours_Studied", "Previous_Scores", "Attendance",
        "Sleep_Hours", "Physical_Activity", "Extracurricular_Activities"
    ],
    "Resources_motivation": [
        "Hours_Studied", "Previous_Scores", "Attendance",
        "Motivation_Level", "Access_to_Resources", "Internet_Access",
        "Parental_Involvement", "Teacher_Quality"
    ],
    "All_features_except_target": [
        c for c in df.columns if c != TARGET_COL
    ]
}

In [12]:

print("Feature Combination Experiments - Test Set Metrics")
print("name\t\t\t\tMAE\t\tMSE\t\tRMSE\t\tR2")
for name, cols in feature_sets.items():
    # Keep only columns that exist in the dataframe (safety)
    cols = [c for c in cols if c in df.columns]
    mae, mse, rmse, r2 = evaluate_feature_set(cols)
    print(f"{name:28s}\t{mae:.4f}\t{mse:.4f}\t{rmse:.4f}\t{r2:.4f}")

Feature Combination Experiments - Test Set Metrics
name				MAE		MSE		RMSE		R2
Baseline_hours_only         	2.4476	10.8559	3.2948	0.2320
Hours_plus_previous_scores  	2.4073	10.5438	3.2471	0.2541
Core_academic               	1.2732	5.0694	2.2515	0.6414
Lifestyle_added             	1.3403	5.3101	2.3044	0.6243
Resources_motivation        	0.9945	4.1835	2.0454	0.7040
All_features_except_target  	0.4524	3.2560	1.8044	0.7696


## Bonus 2: Feature Combination Performance — Commentary

### Overall trend
Performance improves substantially as more informative features are added. The **Hours_Studied-only** model is a useful baseline, but it explains a limited amount of score variation (**R² = 0.2320**). Adding academic, behavioral, and resource-related variables steadily reduces error and increases explained variance.

### Key observations (from your results)

- **Baseline (Hours_Studied only)**  
  - **RMSE = 3.2948**, **R² = 0.2320**  
  - Study hours alone has predictive value, but most score differences come from other factors.

- **Hours + Previous_Scores**  
  - Small improvement: **R² 0.2320 → 0.2541**, **RMSE 3.2948 → 3.2471**  
  - Prior performance is helpful, but this feature set is still too limited to model exam outcomes well.

- **Core_academic (Hours, Previous_Scores, Attendance, Tutoring)**  
  - Major jump: **RMSE = 2.2515**, **R² = 0.6414**  
  - This suggests exam score is strongly tied to *academic history* and *engagement* (attendance/tutoring), not just study time.

- **Lifestyle_added (adds Sleep, Physical Activity, Extracurriculars)**  
  - Slightly worse than Core_academic: **R² = 0.6243** vs **0.6414**, **RMSE = 2.3044** vs **2.2515**  
  - These lifestyle variables may add noise, have weaker linear relationships, or require a different model (nonlinear/interaction effects) to help.

- **Resources_motivation (adds Motivation + Resources + Parent/Teacher factors)**  
  - Strong improvement: **RMSE = 2.0454**, **R² = 0.7040**  
  - Motivation and learning environment clearly contribute meaningful signal beyond academics alone.

- **All_features_except_target (full feature set)**  
  - Best overall: **MAE = 0.4524**, **RMSE = 1.8044**, **R² = 0.7696**  
  - The model explains ~**77%** of variance, indicating the dataset contains strong predictors and that combining them captures exam performance much better than hours alone.

### Conclusion
The experiments show that while `Hours_Studied` is a valid baseline predictor, **multi-factor models** (especially those including academic history, motivation, and resource/parent/teacher context) produce far better accuracy. The best-performing approach in your tests was using **all available features**, achieving the lowest errors and highest R².

### Note (optional, for rigor)
Because **All_features_except_target** includes many variables, it’s good practice to validate with **cross-validation** and consider **regularization** (Ridge/Lasso) to reduce overfitting and improve generalization.